In [5]:
from pyspark.sql import SparkSession

PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Permormance_Analytics") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [6]:
from pyspark.sql.functions import *

spark.sql("""
SELECT 
    COUNT(*) as total_files,
    ROUND(AVG(file_size_in_bytes) / 1024, 2) as avg_size_kb,
    SUM(record_count) as total_records
FROM my_catalog.default.user_events.files
""").show()

spark.sql("""
SELECT 
    file_path, 
    file_size_in_bytes, 
    record_count 
FROM my_catalog.default.user_events.files 
LIMIT 5
""").show(truncate=False)

+-----------+-----------+-------------+
|total_files|avg_size_kb|total_records|
+-----------+-----------+-------------+
|        100|       1.62|         3283|
+-----------+-----------+-------------+

+-----------------------------------------------------------------------------------------------------+------------------+------------+
|file_path                                                                                            |file_size_in_bytes|record_count|
+-----------------------------------------------------------------------------------------------------+------------------+------------+
|s3a://warehouse/default/user_events/data/00000-101-1990a14e-68b9-441b-9bd6-27409ae5b292-00001.parquet|1429              |9           |
|s3a://warehouse/default/user_events/data/00000-100-1990a14e-68b9-441b-9bd6-27409ae5b292-00001.parquet|1827              |50          |
|s3a://warehouse/default/user_events/data/00000-99-1990a14e-68b9-441b-9bd6-27409ae5b292-00001.parquet |1809            

In [16]:
import time

def benchmark_read():
    start_time = time.time()
    spark.sql("""
        SELECT event_type, COUNT(*), SUM(price) 
        FROM my_catalog.default.user_events 
        GROUP BY event_type
    """).collect() # action for spark to actually calculate things
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    # file numbers as of now
    files_count = spark.sql("SELECT COUNT(*) FROM my_catalog.default.user_events.files").collect()[0][0]
    
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING: {execution_time:.4f}")



benchmark_read()

FILES NUMBER: 383 /// TIME EXECUTING: 2.4050
